In [0]:
%pip install torch pillow torchvision

In [0]:
import os
import joblib
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import mlflow
import mlflow.pyfunc
from mlflow.models.signature import infer_signature
from mlflow import MlflowClient
from mlflow.exceptions import RestException

from config import (
    DATASET_DIR, 
    VOLUME_FEATURES_DIR, 
    EXPERIMENT_NAME, 
    get_run_id
)

from dataset import inference_transforms

In [0]:
class OvarianCancerInferencePipeline(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.transform = inference_transforms

        weights = models.ResNet50_Weights.DEFAULT
        backbone = models.resnet50(weights=weights)
        backbone.fc = nn.Identity()
        for param in backbone.parameters():
            param.requires_grad = False
        self.backbone = backbone.to(self.device)
        self.backbone.eval()

        self.var_selector = joblib.load(context.artifacts["var_selector"])
        self.non_colinear_indices = joblib.load(context.artifacts["non_colinear_indices"])
        self.k_best_selector = joblib.load(context.artifacts["k_best_selector"])
        
        self.classifier = mlflow.pyfunc.load_model(context.artifacts['champion_model_dir'])

        self.class_names = None
        if "class_names" in context.artifacts:
            self.class_names = joblib.load(context.artifacts["class_names"])


    def _extract_image_features(self, image_paths):
        tensors = []
        for path in image_paths:
            img = Image.open(path).convert("RGB")
            tensors.append(self.transform(img))
            
        batch_tensor = torch.stack(tensors).to(self.device)
        
        with torch.no_grad():
            embeddings = self.backbone(batch_tensor)
            
        return embeddings.cpu().numpy()
    
    def predict(self, context, model_input):

        if isinstance(model_input, pd.DataFrame):
            if "image_path" in model_input.columns:
                paths = model_input['image_path'].tolist()
            else:
                paths = model_input.iloc[:, 0].tolist()
        elif isinstance(model_input, list):
            paths = model_input
        else:
            print('Error')
        
        raw_features = self._extract_image_features(paths)
        var_features = self.var_selector.transform(raw_features)
        non_corr_features = var_features[:, self.non_colinear_indices]
        optimal_features = self.k_best_selector.transform(non_corr_features)
        
        predictions = self.classifier.predict(optimal_features)

        if hasattr(predictions, "values"):
            predictions = predictions.values.flatten()
        elif isinstance(predictions, np.ndarray):
            predictions = predictions.flatten()
            
        predictions = predictions.astype(int)

        if self.class_name:
            predicted_labels = [self.class_names[p] for p in predictions]
            return pd.DataFrame({
                "predicted_class_idx": predictions,
                "predicted_subtype": predicted_labels
            })
            
        return pd.DataFrame({"predicted_class_idx": predictions})
        


In [0]:
client = mlflow.MlflowClient()
parent_run_id = get_run_id()


experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"tags.mlflow.parentRunId = '{parent_run_id}'",
    order_by=["metrics.val_f1_macro DESC"]
)
best_run_id = runs.iloc[0]["run_id"]

local_artifacts_dir = "/tmp/pyfunc_packaging"
os.makedirs(local_artifacts_dir, exist_ok=True)

champion_model_dir = mlflow.artifacts.download_artifacts(
    run_id=best_run_id,
    artifact_path="model",
    dst_path=local_artifacts_dir
)

var_path = mlflow.artifacts.download_artifacts(
    run_id=parent_run_id, 
    artifact_path="feature_selectors/var_selector.pkl", 
    dst_path=local_artifacts_dir
)
corr_path = mlflow.artifacts.download_artifacts(
    run_id=parent_run_id, 
    artifact_path="feature_selectors/non_colinear_indices.pkl", 
    dst_path=local_artifacts_dir
)
kbest_path = mlflow.artifacts.download_artifacts(
    run_id=parent_run_id, 
    artifact_path="feature_selectors/k_best_selector.pkl", 
    dst_path=local_artifacts_dir
)

class_names = sorted([d for d in os.listdir(DATASET_DIR) if not d.startswith(".")])
class_names_path = f"{local_artifacts_dir}/class_names.pkl"
joblib.dump(class_names, class_names_path)

artifacts = {
    "var_selector": var_path,
    "non_colinear_indices": corr_path,
    "k_best_selector": kbest_path,
    "champion_model_dir": champion_model_dir,
    "class_names": class_names_path
}

In [0]:
input_example = pd.DataFrame({"image_path": ["/tmp/example_image.png"]})
output_example = pd.DataFrame({
    "predicted_class_idx": pd.Series([0], dtype="int64"),
    "predicted_subtype": ["example_subtype"]
})
signature = infer_signature(input_example, output_example)

with mlflow.start_run(run_name="End_to_End_PyFunc_Packaging", nested=True):
    model_info = mlflow.pyfunc.log_model(
        artifact_path="ovarian_cancer_pipeline_e2e",
        python_model=OvarianCancerInferencePipeline(),
        artifacts=artifacts,
        pip_requirements=[
            "torch>=2.0.0",
            "torchvision>=0.15.0",
            "scikit-learn>=1.3.0",
            "xgboost>=1.7.0",
            "pillow>=9.0.0",
            "pandas>=2.0.0",
            "numpy<2.0.0"
        ]
    )

In [0]:
client = MlflowClient()
REGISTERED_MODEL_NAME = "CompVis_OD.data.ovarian_subtype_detector"

reg_model = mlflow.register_model(
    model_uri=model_info.model_uri,
    name=REGISTERED_MODEL_NAME
)
candidate_version = reg_model.version
print(f"Registered new model version: v{candidate_version}")

candidate_run = client.get_run(best_run_id)
candidate_f1 = candidate_run.data.metrics.get("val_f1_macro", 0.0)
print(f"Candidate v{candidate_version} Val Macro F1: {candidate_f1:.4f}")

champion_exists = False
try:
    current_champion = client.get_model_version_by_alias(
        name=REGISTERED_MODEL_NAME, 
        alias="Champion"
    )
    champion_exists = True
except RestException:
    print("No existing Champion alias found. This is the baseline model.")


EPSILON = 0.005 

if not champion_exists:
    
    client.set_registered_model_alias(REGISTERED_MODEL_NAME, "Champion", candidate_version)
    print(f"Promoted v{candidate_version} to @Champion as initial baseline.")

else:
    
    champ_run_id = current_champion.run_id
    champ_run = client.get_run(champ_run_id)
    champion_f1 = champ_run.data.metrics.get("val_f1_macro", 0.0)
    print(f"Current Champion v{current_champion.version} Val Macro F1: {champion_f1:.4f}")

    if candidate_f1 > (champion_f1 + EPSILON):
        
        client.set_registered_model_alias(REGISTERED_MODEL_NAME, "Champion", candidate_version)
        client.set_registered_model_alias(REGISTERED_MODEL_NAME, "Previous_Champion", current_champion.version)
        print(f"SUCCESS: v{candidate_version} beat v{current_champion.version} ({candidate_f1:.4f} > {champion_f1:.4f}). Promoted to @Champion.")
    else:
    
        client.set_registered_model_alias(REGISTERED_MODEL_NAME, "Challenger", candidate_version)
        print(f"RETAINED: Candidate v{candidate_version} ({candidate_f1:.4f}) did not beat Champion v{current_champion.version} ({champion_f1:.4f} + {EPSILON}). Tagged as @Challenger.")